# 08. 오프라인 마케팅 인사이트 및 시나리오 분석

실제 캠페인 이력이 없는 현재 프로젝트에서 가능한 분석만 수행합니다.

- 실제 Test 결과를 이용한 위험도 구간별 이탈률과 Lift
- 예산별 Top 5%·10% 타깃 성과
- 사전 관측 피처 기반 마케팅 후보군 비교
- **가정값**을 이용한 비용·손익분기 시나리오

> 시나리오 결과는 실제 ROI나 캠페인 효과가 아닙니다. 실제 효과는 향후 무작위 실험으로 검증해야 합니다.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
DASH_DATA = Path("../data/dashboard")
MODELS_DIR = Path("../models")
DASH_DATA.mkdir(parents=True, exist_ok=True)

with open(MODELS_DIR / "lightgbm_enhanced_v2_meta.json", encoding="utf-8") as f:
    model_meta = json.load(f)

THRESHOLD = model_meta["threshold"]
pred = pd.read_csv(PROCESSED_DIR / "frontend_customer_predictions.csv")
ltv = pd.read_csv(DASH_DATA / "customer_ltv_segments.csv")

assert len(pred) == 148940
assert pred["split"].eq("test").all()
assert pred["msno"].duplicated().sum() == 0
assert "revenue_tier" in ltv.columns

df = pred.merge(
    ltv[["msno", "risk_tier", "revenue_tier", "segment", "avg_monthly_revenue"]],
    on="msno", how="inner", validate="one_to_one",
)
assert len(df) == 148940
print(f"Test 고객: {len(df):,}명 / 실제 이탈률: {df['is_churn'].mean():.2%}")
print(f"Enhanced v2 threshold: {THRESHOLD:.6f}")

In [ ]:
# 대형 v2 테이블은 필요한 컬럼만 청크로 읽어 Test 행만 남깁니다.
extra_cols = [
    "msno", "split", "expiry_alignment_abs_gap",
    "activity_change_7_30", "has_log_activity",
]
extra_chunks = []
for chunk in pd.read_csv(
    PROCESSED_DIR / "model_table_enhanced_v2.csv",
    usecols=extra_cols, chunksize=200_000, low_memory=False,
):
    selected = chunk.loc[chunk["split"].eq("test")].copy()
    if len(selected):
        extra_chunks.append(selected)

extra = pd.concat(extra_chunks, ignore_index=True).drop(columns="split")
df = df.merge(extra, on="msno", how="inner", validate="one_to_one")
assert len(df) == 148940
print("추가 마케팅 후보 피처 결합 완료")

## 1. 위험도 10분위별 실제 이탈률

확률 동점이 있어도 각 구간의 고객 수가 동일하도록 순위를 먼저 만든 뒤 10개 구간으로 나눕니다.

In [ ]:
rank_desc = df["churn_probability"].rank(method="first", ascending=False)
df["risk_decile"] = pd.qcut(rank_desc, 10, labels=range(1, 11)).astype(int)

decile = (
    df.groupby("risk_decile", observed=True)
    .agg(
        customers=("msno", "size"),
        churners=("is_churn", "sum"),
        churn_rate=("is_churn", "mean"),
        avg_predicted_risk=("churn_probability", "mean"),
    )
    .reset_index()
)
decile["lift"] = decile["churn_rate"] / df["is_churn"].mean()
decile["churn_capture_pct"] = decile["churners"] / df["is_churn"].sum() * 100
decile["cumulative_capture_pct"] = decile["churn_capture_pct"].cumsum()
decile.to_csv(DASH_DATA / "risk_decile_performance.csv", index=False)
display(decile.style.format({
    "churn_rate": "{:.2%}", "avg_predicted_risk": "{:.2%}",
    "lift": "{:.2f}x", "churn_capture_pct": "{:.1f}%",
    "cumulative_capture_pct": "{:.1f}%",
}))

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.bar(decile["risk_decile"].astype(str), decile["churn_rate"] * 100, color="#2a78d6")
ax.axhline(df["is_churn"].mean() * 100, color="#e34948", linestyle="--", label="전체 이탈률")
ax.set_xlabel("위험도 분위 (1=가장 위험)")
ax.set_ylabel("실제 이탈률 (%)")
ax.set_title("위험도 10분위별 실제 이탈률")
ax.legend()
plt.tight_layout()
plt.show()

## 2. 예산·행동 조건별 타깃 후보 비교

Top 5%·10%는 예측 확률을 정렬해 정확한 고객 수로 자릅니다. 행동 후보군은 모두 1월 31일까지 관측된 정보로 정의합니다.

In [ ]:
ordered_index = df["churn_probability"].sort_values(ascending=False).index
top5_mask = df.index.isin(ordered_index[:round(len(df) * 0.05)])
top10_mask = df.index.isin(ordered_index[:round(len(df) * 0.10)])

candidate_masks = {
    "위험도 상위 5%": top5_mask,
    "위험도 상위 10%": top10_mask,
    "threshold 이상 전체": df["churn_probability"].ge(THRESHOLD),
    "고위험 + 자동갱신 OFF": df["churn_probability"].ge(THRESHOLD) & df["last_is_auto_renew"].eq(0),
    "고위험 + 만료 정합성 7일 초과": df["churn_probability"].ge(THRESHOLD) & df["expiry_alignment_abs_gap"].gt(7),
    "고위험 + 최근 로그 30일 초과": (
        df["churn_probability"].ge(THRESHOLD)
        & df["has_log_activity"].eq(1)
        & df["days_since_last_log"].gt(30)
    ),
    "고위험 + 최근 활동률 급감": df["churn_probability"].ge(THRESHOLD) & df["activity_change_7_30"].le(-0.20),
    "고위험/고월매출": df["segment"].eq("고위험/고월매출"),
}

rows = []
base_rate = df["is_churn"].mean()
for strategy, mask in candidate_masks.items():
    group = df.loc[mask]
    rows.append({
        "strategy": strategy,
        "customers": len(group),
        "customer_pct": len(group) / len(df) * 100,
        "churners": int(group["is_churn"].sum()),
        "churn_rate": group["is_churn"].mean(),
        "lift": group["is_churn"].mean() / base_rate,
        "recall": group["is_churn"].sum() / df["is_churn"].sum(),
        "avg_monthly_revenue": group["avg_monthly_revenue"].mean(),
    })
target_summary = pd.DataFrame(rows)
target_summary.to_csv(DASH_DATA / "offline_marketing_targets.csv", index=False)
display(target_summary.style.format({
    "customer_pct": "{:.2f}%", "churn_rate": "{:.2%}",
    "lift": "{:.2f}x", "recall": "{:.2%}",
    "avg_monthly_revenue": "{:,.1f}",
}))

## 3. 가정 기반 비용·손익분기 시나리오

아래 값은 실제 캠페인 성과가 아니라 의사결정 구조를 보여주기 위한 가정입니다. `prevented_churn_rate`는 캠페인이 없었다면 이탈했을 고객 중 방어되는 비율입니다.

In [ ]:
# 필요에 따라 가정값을 직접 변경합니다.
CONTACT_COST_PER_TARGET = 2.0       # NTD
INCENTIVE_COST = 30.0               # 혜택 사용 1건당 NTD
INCENTIVE_REDEMPTION_RATE = 0.20    # 대상 고객 중 혜택 사용 가정
RETAINED_MONTHS = 3                 # 방어 고객의 추가 잔존 개월 가정
GROSS_MARGIN_RATE = 0.70            # 매출총이익률 가정
EFFECT_SCENARIOS = [0.05, 0.10, 0.15]

scenario_rows = []
for row in target_summary.itertuples(index=False):
    campaign_cost = (
        row.customers * CONTACT_COST_PER_TARGET
        + row.customers * INCENTIVE_REDEMPTION_RATE * INCENTIVE_COST
    )
    value_per_saved_customer = row.avg_monthly_revenue * RETAINED_MONTHS * GROSS_MARGIN_RATE
    break_even_effect = (
        campaign_cost / (row.churners * value_per_saved_customer)
        if row.churners > 0 and value_per_saved_customer > 0 else np.nan
    )
    for effect in EFFECT_SCENARIOS:
        saved_customers = row.churners * effect
        expected_gross_value = saved_customers * value_per_saved_customer
        scenario_rows.append({
            "strategy": row.strategy,
            "assumed_prevented_churn_rate": effect,
            "selected_customers": row.customers,
            "historical_churners": row.churners,
            "assumed_saved_customers": saved_customers,
            "campaign_cost_ntd": campaign_cost,
            "assumed_gross_value_ntd": expected_gross_value,
            "assumed_net_value_ntd": expected_gross_value - campaign_cost,
            "break_even_prevented_rate": break_even_effect,
        })

roi_scenarios = pd.DataFrame(scenario_rows)
roi_scenarios.to_csv(DASH_DATA / "marketing_roi_scenarios.csv", index=False)
display(roi_scenarios.style.format({
    "assumed_prevented_churn_rate": "{:.0%}",
    "assumed_saved_customers": "{:,.1f}",
    "campaign_cost_ntd": "{:,.0f}",
    "assumed_gross_value_ntd": "{:,.0f}",
    "assumed_net_value_ntd": "{:,.0f}",
    "break_even_prevented_rate": "{:.1%}",
}))

## 4. 해석 원칙

- 위험도 구간별 이탈률과 Lift는 과거 Test 데이터에서 실제 계산한 값입니다.
- ROI 표의 효과율·비용·마진·추가 잔존 개월은 모두 사용자가 입력한 가정입니다.
- 과거 이탈 고객이 캠페인으로 방어된다는 보장은 없습니다.
- 자동갱신, 만료 정합성, 최근 활동은 인과 원인이 아니라 예측 신호입니다.
- 실제 마케팅 효과와 ROI는 실서비스의 랜덤 실험 또는 적절한 인과추론 데이터가 있어야 계산할 수 있습니다.
- 이 분석은 캠페인 예산과 실험 우선순위를 정하기 위한 오프라인 의사결정 보조 자료입니다.